In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

# Métodos

In [2]:
def delete_hdf_table(file_path, key):
    with pd.HDFStore(file_path, mode='a') as store:
        if f'/{key}' in store.keys():
            store.remove(f'/{key}')
            print(f"Table '{key}' removed from HDF5.")
        else:
            print(f"Table '{key}' not found in HDF5.")

def read_hdf(file_path, key):
    return pd.read_hdf(file_path, mode = 'r', key=key, encoding='utf-8')

def show_hdf_tables(file_path):
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print(f"Current tables in {file_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

def write_hdf_chunks(df, path, key, chunksize=10_000_000):
    with pd.HDFStore(path, mode='a', complevel=9, complib='blosc:lz4') as store: # Open in read+write mode to append
        for i in range(0, len(df), chunksize):
            chunk = df.iloc[i:i+chunksize]
            append_mode = i != 0 # Append after the first chunk to avoid overwriting data from previous chunks
            store.append(
                key,
                chunk,
                format='table',
                data_columns=True,
                min_itemsize={'gauge_code': 20},
                encoding='utf-8',
                append=append_mode
            )
            del chunk  # free memory
            print(f"Written rows {i} to {min(i+chunksize-1, len(df))}")

def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    """
    Read HDF file in chunks and concatenate into a single DataFrame.
    Only works if the HDF key is stored in 'table' format.
    
    Parameters:
    file_path (str): Path to the HDF5 file
    key (str): Key/table name to read
    chunksize (int): Number of rows per chunk
    
    Returns:
    pd.DataFrame: Concatenated DataFrame
    """
    try:
        with pd.HDFStore(file_path, mode='r+') as store:  # Changed to 'r+' (read/write) and automatically closes the store
            if store.get_storer(key).is_table:
                dfs = []
                i = 1
                for chunk in store.select(key, chunksize=chunksize):
                    dfs.append(chunk)
                    print(f"Chunk {i} with {len(chunk)} rows read.")
                    i += 1                
                if dfs:  # Check if any chunks were read
                    return pd.concat(dfs, ignore_index=True)
                else:
                    print("No data found or chunksize too large.")
                    return pd.DataFrame()
            else:
                # If not table format, read all at once
                print("Not in table format, reading all data at once.")
                return store.select(key)  # Convert to DataFrame
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return pd.DataFrame()
    except KeyError:
        print(f"Key {key} not found in file {file_path}.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading HDF file: {e}")
        return pd.DataFrame()

# Data processing

In [3]:
cleaned_path = r'./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'

In [4]:
df_data = read_hdf_chunks(cleaned_path, key='table_data_outlier_filtered', chunksize=10_000_000)
df_data['month'] = df_data['datetime'].dt.month
df_data['year'] = df_data['datetime'].dt.year
df_data

Chunk 1 with 1000000 rows read.
Chunk 2 with 1000000 rows read.
Chunk 3 with 1000000 rows read.
Chunk 4 with 1000000 rows read.
Chunk 5 with 1000000 rows read.
Chunk 6 with 1000000 rows read.
Chunk 7 with 1000000 rows read.
Chunk 8 with 1000000 rows read.
Chunk 9 with 1000000 rows read.
Chunk 10 with 1000000 rows read.
Chunk 11 with 1000000 rows read.
Chunk 12 with 1000000 rows read.
Chunk 13 with 1000000 rows read.
Chunk 14 with 1000000 rows read.
Chunk 15 with 1000000 rows read.
Chunk 16 with 1000000 rows read.
Chunk 17 with 1000000 rows read.
Chunk 18 with 1000000 rows read.
Chunk 19 with 1000000 rows read.
Chunk 20 with 1000000 rows read.
Chunk 21 with 1000000 rows read.
Chunk 22 with 1000000 rows read.
Chunk 23 with 1000000 rows read.
Chunk 24 with 1000000 rows read.
Chunk 25 with 1000000 rows read.
Chunk 26 with 1000000 rows read.
Chunk 27 with 1000000 rows read.
Chunk 28 with 1000000 rows read.
Chunk 29 with 1000000 rows read.
Chunk 30 with 1000000 rows read.
Chunk 31 with 10000

,gauge_code,datetime,rain_mm,month,year
0,110018901A,2021-02-02,9.48,2,2021
1,110018901A,2021-02-03,4.73,2,2021
2,110018901A,2021-02-04,73.28,2,2021
3,110018901A,2021-02-07,0.20,2,2021
4,110018901A,2021-02-08,0.00,2,2021
...,...,...,...,...,...
123590252,88690050,2023-12-27,0.00,12,2023
123590253,88690050,2023-12-28,0.00,12,2023
123590254,88690050,2023-12-29,2.00,12,2023
123590255,88690050,2023-12-30,0.00,12,2023


In [5]:
def calculateQ3(df_monthly_thresholds):
    # df_monthly_thresholds = df.copy(deep = True)  # Create a deep copy of the DataFrame
    # df_monthly_thresholds['month'] = df_monthly_thresholds['datetime'].dt.month
    # df_monthly_thresholds['year'] = df_monthly_thresholds['datetime'].dt.year

    df_monthly_thresholds = df_monthly_thresholds[df_monthly_thresholds['rain_mm'] >= 1.0]  # Filter rainy days first
    df_monthly_thresholds = df_monthly_thresholds.groupby(['gauge_code', 'month'])['rain_mm'].agg(
        Q1=lambda x: x.quantile(0.25) if not x.empty else np.nan,
        Q3=lambda x: x.quantile(0.75) if not x.empty else np.nan
    ).reset_index()

    df_monthly_thresholds['IQR'] = df_monthly_thresholds['Q3'] - df_monthly_thresholds['Q1']
    # df_monthly_thresholds['lower_bound'] = df_monthly_thresholds['Q1'] - 1.5 * df_monthly_thresholds['IQR']
    df_monthly_thresholds['upper_bound'] = df_monthly_thresholds['Q3'] + 1.5 * df_monthly_thresholds['IQR']

    return df_monthly_thresholds

df_monthly_thresholds = calculateQ3(df_data)
df_monthly_thresholds

,gauge_code,month,Q1,Q3,IQR,upper_bound
0,00047000,1,5.60,13.70,8.10,25.850
1,00047000,2,6.20,33.30,27.10,73.950
2,00047000,3,4.65,45.00,40.35,105.525
3,00047000,4,4.60,11.20,6.60,21.100
4,00047000,5,4.60,9.60,5.00,17.100
...,...,...,...,...,...,...
206878,S716,6,3.70,16.10,12.40,34.700
206879,S716,7,1.00,1.00,0.00,1.000
206880,S716,10,7.00,31.00,24.00,67.000
206881,S716,11,5.60,24.00,18.40,51.600


In [6]:
df_data = df_data.merge(df_monthly_thresholds[['gauge_code', 'month', 'upper_bound']], on=['gauge_code', 'month'], how='left')
df_data

,gauge_code,datetime,rain_mm,month,year,upper_bound
0,110018901A,2021-02-02,9.48,2,2021,47.195
1,110018901A,2021-02-03,4.73,2,2021,47.195
2,110018901A,2021-02-04,73.28,2,2021,47.195
3,110018901A,2021-02-07,0.20,2,2021,47.195
4,110018901A,2021-02-08,0.00,2,2021,47.195
...,...,...,...,...,...,...
123590252,88690050,2023-12-27,0.00,12,2023,34.250
123590253,88690050,2023-12-28,0.00,12,2023,34.250
123590254,88690050,2023-12-29,2.00,12,2023,34.250
123590255,88690050,2023-12-30,0.00,12,2023,34.250


In [7]:
df_data['upper_bound'] = df_data['upper_bound'].fillna(0)  # Fill NaN values with 0
df_data

,gauge_code,datetime,rain_mm,month,year,upper_bound
0,110018901A,2021-02-02,9.48,2,2021,47.195
1,110018901A,2021-02-03,4.73,2,2021,47.195
2,110018901A,2021-02-04,73.28,2,2021,47.195
3,110018901A,2021-02-07,0.20,2,2021,47.195
4,110018901A,2021-02-08,0.00,2,2021,47.195
...,...,...,...,...,...,...
123590252,88690050,2023-12-27,0.00,12,2023,34.250
123590253,88690050,2023-12-28,0.00,12,2023,34.250
123590254,88690050,2023-12-29,2.00,12,2023,34.250
123590255,88690050,2023-12-30,0.00,12,2023,34.250


In [8]:
df_data.pop('month')
# df_data.pop('year')

0             2
1             2
2             2
3             2
4             2
             ..
123590252    12
123590253    12
123590254    12
123590255    12
123590256    12
Name: month, Length: 123590257, dtype: int32

In [9]:
df_data['outlier'] = (df_data['rain_mm'] > df_data['upper_bound']).astype(np.uint8)
# df_data['year'] = df_data['datetime'].dt.year
df_data

,gauge_code,datetime,rain_mm,year,upper_bound,outlier
0,110018901A,2021-02-02,9.48,2021,47.195,0
1,110018901A,2021-02-03,4.73,2021,47.195,0
2,110018901A,2021-02-04,73.28,2021,47.195,1
3,110018901A,2021-02-07,0.20,2021,47.195,0
4,110018901A,2021-02-08,0.00,2021,47.195,0
...,...,...,...,...,...,...
123590252,88690050,2023-12-27,0.00,2023,34.250,0
123590253,88690050,2023-12-28,0.00,2023,34.250,0
123590254,88690050,2023-12-29,2.00,2023,34.250,0
123590255,88690050,2023-12-30,0.00,2023,34.250,0


In [10]:
df_data = df_data[['gauge_code', 'year', 'outlier']].copy(deep = True)
df_data

,gauge_code,year,outlier
0,110018901A,2021,0
1,110018901A,2021,0
2,110018901A,2021,1
3,110018901A,2021,0
4,110018901A,2021,0
...,...,...,...
123590252,88690050,2023,0
123590253,88690050,2023,0
123590254,88690050,2023,0
123590255,88690050,2023,0


In [11]:
df_q3_outliers = df_data.groupby(['gauge_code', 'year']).agg(
    count_outliers=('outlier', lambda x: x.sum() if not x.empty else 0),  # Count the number of outliers per gauge_code and year
    active_days=('outlier', 'count')  # Count the number of datetime entries per gauge_code and year
).reset_index()
df_q3_outliers

,gauge_code,year,count_outliers,active_days
0,00047000,1961,7,365
1,00047000,1962,2,365
2,00047000,1963,1,365
3,00047000,1964,1,366
4,00047002,1977,2,23
...,...,...,...,...
345863,S713,2021,2,365
345864,S714,2021,7,365
345865,S715,2021,8,365
345866,S716,2021,4,365


In [12]:
df_q3_outliers['outlier_percentage'] = df_q3_outliers['count_outliers'] / df_q3_outliers['active_days'] * 100  # Calculate the percentage of outliers
df_q3_outliers['q3_outliers'] = 100 - df_q3_outliers['outlier_percentage']  # Calculate the percentage of non-outliers
df_q3_outliers

,gauge_code,year,count_outliers,active_days,outlier_percentage,q3_outliers
0,00047000,1961,7,365,1.917808,98.082192
1,00047000,1962,2,365,0.547945,99.452055
2,00047000,1963,1,365,0.273973,99.726027
3,00047000,1964,1,366,0.273224,99.726776
4,00047002,1977,2,23,8.695652,91.304348
...,...,...,...,...,...,...
345863,S713,2021,2,365,0.547945,99.452055
345864,S714,2021,7,365,1.917808,98.082192
345865,S715,2021,8,365,2.191781,97.808219
345866,S716,2021,4,365,1.095890,98.904110


In [ ]:
df_q3_outliers = df_q3_outliers[['gauge_code', 'year', 'q3_outliers']].copy(deep = True)
# df_q3_outliers.to_hdf(cleaned_path
#                   , key = 'table_q3_outliers'
#                   , encoding = 'utf-8'
#                   , mode='r+'
#                   , append = False
#                   , complevel=9
#                   , format='table')
write_hdf_chunks(df_q3_outliers, cleaned_path, key = 'table_q3_outliers')

Written rows 0 to 345868


NameError: name 'show_hdf_chunks' is not defined

In [15]:
show_hdf_tables(cleaned_path)

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_p_availability
  - table_preclassif
  - table_q1_gaps
  - table_q2_week
  - table_q3_outliers


In [16]:
df_q3_outliers=read_hdf(cleaned_path, key = 'table_q3_outliers')
df_q3_outliers

,gauge_code,year,q3_outliers
0,00047000,1961,98.082192
1,00047000,1962,99.452055
2,00047000,1963,99.726027
3,00047000,1964,99.726776
4,00047002,1977,91.304348
...,...,...,...
345863,S713,2021,99.452055
345864,S714,2021,98.082192
345865,S715,2021,97.808219
345866,S716,2021,98.904110
